TASK-1: Clean nav_history.csv

In [1]:
import pandas as pd
import numpy as np
import sqlite3
from sqlalchemy import create_engine

In [3]:
nav=pd.read_csv('../data/raw/02_nav_history.csv')

In [4]:
nav.head()

,amfi_code,date,nav
0,119551,2022-01-03,54.3856
1,119551,2022-01-04,54.3474
2,119551,2022-01-05,54.6869
3,119551,2022-01-06,55.4550
4,119551,2022-01-07,55.3692


In [5]:
nav.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 46000 entries, 0 to 45999
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   amfi_code  46000 non-null  int64  
 1   date       46000 non-null  object 
 2   nav        46000 non-null  float64
dtypes: float64(1), int64(1), object(1)
memory usage: 1.1+ MB


In [6]:
nav.isnull().sum()

amfi_code    0
date         0
nav          0
dtype: int64

In [7]:
nav['date'] = pd.to_datetime(
    nav['date']
)

In [8]:
nav = nav.sort_values(
    ['amfi_code','date']
)

In [9]:
nav = nav.drop_duplicates()

In [10]:
nav.duplicated().sum()

0

In [11]:
nav[nav['nav'] <= 0]

,amfi_code,date,nav


In [12]:
nav.isnull().sum()

amfi_code    0
date         0
nav          0
dtype: int64

In [13]:
nav['nav'] = nav.groupby(
    'amfi_code'
)['nav'].ffill()

In [18]:
import os

os.makedirs("../data/processed", exist_ok=True)
nav.to_csv("../data/processed/clean_nav.csv", index=False)

TASK-2: Clean investor_transactions.csv

In [20]:
tx = pd.read_csv(
    "../data/raw/08_investor_transactions.csv"
)

In [21]:
tx.head()

,investor_id,transaction_date,amfi_code,transaction_type,amount_inr,state,city,city_tier,age_group,gender,annual_income_lakh,payment_mode,kyc_status
0,INV003054,2024-01-01,119092,SIP,1834,Telangana,Hyderabad,T30,56+,Female,77.1,UPI,Verified
1,INV002952,2024-01-01,148567,Redemption,392882,Punjab,Amritsar,B30,18-25,Male,7.1,Cheque,Verified
2,INV003420,2024-01-01,118636,SIP,912,Haryana,Faridabad,B30,36-45,Male,47.2,Mandate,Verified
3,INV003436,2024-01-01,118634,SIP,1102,Maharashtra,Mumbai,T30,36-45,Female,54.4,Cheque,Pending
4,INV004691,2024-01-01,119094,Lumpsum,8682,Delhi,Noida,T30,26-35,Male,14.5,Net Banking,Pending


In [22]:
tx.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32778 entries, 0 to 32777
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   investor_id         32778 non-null  object 
 1   transaction_date    32778 non-null  object 
 2   amfi_code           32778 non-null  int64  
 3   transaction_type    32778 non-null  object 
 4   amount_inr          32778 non-null  int64  
 5   state               32778 non-null  object 
 6   city                32778 non-null  object 
 7   city_tier           32778 non-null  object 
 8   age_group           32778 non-null  object 
 9   gender              32778 non-null  object 
 10  annual_income_lakh  32778 non-null  float64
 11  payment_mode        32778 non-null  object 
 12  kyc_status          32778 non-null  object 
dtypes: float64(1), int64(2), object(10)
memory usage: 3.3+ MB


In [23]:
tx['transaction_type'].unique()

array(['SIP', 'Redemption', 'Lumpsum'], dtype=object)

In [24]:
tx['transaction_type'] = (
    tx['transaction_type']
    .str.strip()
    .str.title()
)

In [25]:
tx[tx['amount_inr'] <= 0]

,investor_id,transaction_date,amfi_code,transaction_type,amount_inr,state,city,city_tier,age_group,gender,annual_income_lakh,payment_mode,kyc_status


In [26]:
tx = tx[
    tx['amount_inr'] > 0
]

In [27]:
tx['transaction_date'] = pd.to_datetime(
    tx['transaction_date']
)

In [28]:
tx['kyc_status'].value_counts()

kyc_status
Verified    30146
Pending      2632
Name: count, dtype: int64

In [29]:
tx.to_csv(
    "../data/processed/clean_transactions.csv",
    index=False
)

TASK-3: Clean scheme_performance.csv

In [30]:
perf = pd.read_csv(
    "../data/raw/07_scheme_performance.csv"
)

In [31]:
perf.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40 entries, 0 to 39
Data columns (total 19 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   amfi_code           40 non-null     int64  
 1   scheme_name         40 non-null     object 
 2   fund_house          40 non-null     object 
 3   category            40 non-null     object 
 4   plan                40 non-null     object 
 5   return_1yr_pct      40 non-null     float64
 6   return_3yr_pct      40 non-null     float64
 7   return_5yr_pct      40 non-null     float64
 8   benchmark_3yr_pct   40 non-null     float64
 9   alpha               40 non-null     float64
 10  beta                40 non-null     float64
 11  sharpe_ratio        40 non-null     float64
 12  sortino_ratio       40 non-null     float64
 13  std_dev_ann_pct     40 non-null     float64
 14  max_drawdown_pct    40 non-null     float64
 15  aum_crore           40 non-null     int64  
 16  expense_ra

In [32]:
perf['return_1yr_pct'] = pd.to_numeric(
    perf['return_1yr_pct'],
    errors='coerce'
)

In [33]:
perf[
    perf['sharpe_ratio'] < 0
]

,amfi_code,scheme_name,fund_house,category,plan,return_1yr_pct,return_3yr_pct,return_5yr_pct,benchmark_3yr_pct,alpha,beta,sharpe_ratio,sortino_ratio,std_dev_ann_pct,max_drawdown_pct,aum_crore,expense_ratio_pct,morningstar_rating,risk_grade


In [34]:
perf[
    (perf['expense_ratio_pct'] < 0.1)
    |
    (perf['expense_ratio_pct'] > 2.5)
]

,amfi_code,scheme_name,fund_house,category,plan,return_1yr_pct,return_3yr_pct,return_5yr_pct,benchmark_3yr_pct,alpha,beta,sharpe_ratio,sortino_ratio,std_dev_ann_pct,max_drawdown_pct,aum_crore,expense_ratio_pct,morningstar_rating,risk_grade


In [35]:
perf.to_csv(
    "../data/processed/clean_performance.csv",
    index=False
)

TASK-4: Create SQLite Database and Create Schema

In [36]:
conn = sqlite3.connect(
    "../data/db/bluestock_mf.db"
)

Task 5: Load Data into SQLite

In [37]:
engine = create_engine(
    "sqlite:///../data/db/bluestock_mf.db"
)

In [38]:
fund_master = pd.read_csv(
    "../data/raw/01_fund_master.csv"
)

fund_master.to_sql(
    "dim_fund",
    engine,
    if_exists="replace",
    index=False
)

40

In [39]:
nav.to_sql(
    "fact_nav",
    engine,
    if_exists="replace",
    index=False
)

46000

In [40]:
tx.to_sql(
    "fact_transactions",
    engine,
    if_exists="replace",
    index=False
)

32778

In [41]:
perf.to_sql(
    "fact_performance",
    engine,
    if_exists="replace",
    index=False
)

40

In [42]:
pd.read_sql(
    "SELECT * FROM dim_fund LIMIT 5",
    engine
)

,amfi_code,fund_house,scheme_name,category,sub_category,plan,launch_date,benchmark,expense_ratio_pct,exit_load_pct,min_sip_amount,min_lumpsum_amount,fund_manager,risk_category,sebi_category_code
0,119551,SBI Mutual Fund,SBI Bluechip Fund - Regular Plan - Growth,Equity,Large Cap,Regular,2006-02-14,NIFTY 100 TRI,1.54,1.0,500,1000,Sohini Andani,Moderate,EC01
1,119552,SBI Mutual Fund,SBI Bluechip Fund - Direct Plan - Growth,Equity,Large Cap,Direct,2013-01-01,NIFTY 100 TRI,0.66,1.0,500,1000,Sohini Andani,Moderate,EC01
2,119598,SBI Mutual Fund,SBI Small Cap Fund - Regular Plan - Growth,Equity,Small Cap,Regular,2009-09-09,BSE 250 SmallCap TRI,1.43,1.0,500,1000,R. Srinivasan,Very High,EC03
3,119599,SBI Mutual Fund,SBI Small Cap Fund - Direct Plan - Growth,Equity,Small Cap,Direct,2013-01-01,BSE 250 SmallCap TRI,0.72,1.0,500,1000,R. Srinivasan,Very High,EC03
4,119120,SBI Mutual Fund,SBI Magnum Gilt Fund - Regular Plan - Growth,Debt,Gilt,Regular,2000-12-30,CRISIL Dynamic Gilt Index,0.77,0.0,500,1000,Dinesh Ahuja,Low,DC02


In [43]:
pd.read_sql(
    "SELECT COUNT(*) FROM dim_fund",
    engine
)

,COUNT(*)
0,40


In [44]:
pd.read_sql(
    "SELECT COUNT(*) FROM fact_nav",
    engine
)

,COUNT(*)
0,46000


In [45]:
pd.read_sql(
    "SELECT COUNT(*) FROM fact_transactions",
    engine
)

,COUNT(*)
0,32778


In [46]:
pd.read_sql(
    "SELECT COUNT(*) FROM fact_performance",
    engine
)

,COUNT(*)
0,40
